<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/memory/RecollectMemory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recollect Memory

[Recollect](https://github.com/cobusgreyling/recollect) is a self-hosted long-term memory layer for AI agents. It extracts durable facts from conversations, scopes them per user/session, and retrieves them with hybrid search.

This notebook shows how to use `RecollectMemory` with LlamaIndex chat engines and workflow agents.

If you're opening this Notebook on colab, you will probably need to install LlamaIndex.

In [ ]:
%pip install llama-index llama-index-memory-recollect recollect-ai

## Setup with Recollect (local, no API keys)

Use `RecollectConfig.local_dev()` for SQLite storage and a deterministic local embedder—ideal for prototyping without cloud credentials.

In [ ]:
from llama_index.memory.recollect import RecollectMemory
from recollect.config import RecollectConfig

context = {"user_id": "alice"}
memory = RecollectMemory.from_config(
    context=context,
    config=RecollectConfig.local_dev().model_dump(),
    search_msg_limit=4,
)

`context` must include at least one of `user_id`, `agent_id`, or `run_id`.

`search_msg_limit` controls how many recent chat messages are concatenated into the retrieval query (default `5`).

### Initialize LLM

In [ ]:
import os

from llama_index.llms.openai import OpenAI

os.environ["OPENAI_API_KEY"] = "sk-..."
llm = OpenAI(model="gpt-4o-mini")

## Recollect for Function Calling Agents

In [ ]:
def call_fn(name: str):
    """Call the provided name."""
    print(f"Calling... {name}")


def email_fn(name: str):
    """Email the provided name."""
    print(f"Emailing... {name}")

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent

agent = FunctionAgent(tools=[email_fn, call_fn], llm=llm)

In [ ]:
response = await agent.run("Hi, my name is Mayank.", memory=memory)
print(str(response))

In [ ]:
response = await agent.run(
    "My preferred way of communication is email.", memory=memory
)
print(str(response))

## SimpleChatEngine

In [ ]:
from llama_index.core import SimpleChatEngine

chat_engine = SimpleChatEngine.from_defaults(llm=llm, memory=memory)
print(chat_engine.chat("What do you remember about me?"))

## References

- [Recollect GitHub](https://github.com/cobusgreyling/recollect)
- [LangChain integration guide](https://github.com/cobusgreyling/recollect/blob/main/docs/integrations/langchain.md)